<a href="https://colab.research.google.com/github/Bpriya42/DeepLearning/blob/main/1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

In [ ]:
import os
os.chdir('/content/drive/My Drive/ColabNotebooks/Project')

In [ ]:
!pip install textblob
!pip install vaderSentiment

In [ ]:
import pandas as pd

# Load the data
data = pd.read_csv('Dataset/Combined_News_DJIA(train).csv')

In [ ]:
# Define a function to clean the text
def clean_text(text):
    return text.replace("b'", "").replace("b\"", "").lower().strip("'\"")
# List of columns from Top1 to Top25
top_columns = [f'Top{i}' for i in range(1, 26)]

# Apply the cleaning function to each column directly
for col in top_columns:
    data[col] = data[col].astype(str).apply(clean_text)


In [ ]:
print(data['Top1'][3])

In [ ]:
print(data['Top1'][6])

In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# Initialize VADER
analyzer = SentimentIntensityAnalyzer()

# Function to calculate sentiment score
def get_vader_sentiment(text):
    return analyzer.polarity_scores(text)['compound']  # compound score as an overall sentiment score


In [ ]:
# Apply VADER sentiment analysis

average_sentiments = []
for index, row in data.iterrows():
    # Calculate sentiment for each headline and store in a list
    sentiment_scores = [get_vader_sentiment(row[col]) for col in top_columns]
    # Calculate the average sentiment score for the row
    average_sentiment = sum(sentiment_scores) / len(sentiment_scores)
    # Append the average sentiment score to the list
    average_sentiments.append(average_sentiment)

# Add the average_sentiment column to the DataFrame
data['average_sentiment'] = average_sentiments

# Verify the results
print(data.head())

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Convert 'Date' column to datetime format for proper plotting
data['Date'] = pd.to_datetime(data['Date'])

# Sort data by date in case it's not in chronological order
data = data.sort_values(by='Date')

# Calculate the 30-day moving average for sentiment
data['Sentiment_Moving_Avg'] = data['average_sentiment'].rolling(window=30).mean()

# Plot the smoothed sentiment over time
plt.figure(figsize=(12, 6))
plt.plot(data['Date'], data['Sentiment_Moving_Avg'], color='blue', label='30-Day Moving Average Sentiment')
plt.xlabel('Date')
plt.ylabel('Average Sentiment Score')
plt.title('Sentiment Trend Over Time (30-Day Moving Average)')
plt.legend()
plt.grid(True)
plt.show()



In [ ]:
# Categorize each row based on average sentiment score
data['Sentiment_Category'] = data['average_sentiment'].apply(lambda x: 'Positive' if x > 0 else ('Negative' if x < 0 else 'Neutral'))

# Count the occurrences of each sentiment category
sentiment_counts = data['Sentiment_Category'].value_counts()
print(sentiment_counts)

# Plot the sentiment counts
plt.figure(figsize=(8, 6))
sentiment_counts.plot(kind='bar', color=['green', 'red', 'gray'])
plt.xlabel('Sentiment Category')
plt.ylabel('Count')
plt.title('Count of Positive, Negative, and Neutral Sentiments')
plt.show()


In [ ]:
data.head()